In [25]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
from tensorboard import notebook

%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [2]:
tf.__version__

'2.1.0'

## Загрузка данных

In [3]:
train_df = pd.read_csv('datas/project/fashion-mnist_train.csv')
test_df = pd.read_csv('datas/project/fashion-mnist_test.csv')

In [4]:
# разбитие по переменным 
y_train, x_train = train_df.iloc[:,0], train_df.iloc[:,1:]
y_test, x_test = test_df.iloc[:,0], test_df.iloc[:,1:]
# нормализация признаков
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)
# представление ответов в One-Hot Encoding
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)

In [5]:
y_train.shape, x_train.shape

((60000, 10), (60000, 784))

## Логистическая регрессия

Для решения задачи классификации предлагается начать с использования логистической регрессии. В данном случае, количество признаков равно 28x28=784, так же мы имеем 60000 объектов в тренировочной выборке. Поэтому рекомендуется использовать tensorflow или keras для выполнения этого задания. Используйте стохастический градиентный спуск (stochastic gradient descent) в качестве алгоритма оптимизации.

По своей сути, логистическая регрессия может быть реализована как нейронная сеть без скрытых слоев. В выходном слое содержится количество нейронов, равное количеству классов. В качестве функции активации выходного слоя следует использовать softmax.

Обучите логистическую регрессию на тренировочной выборке и оцените качество на тестовой выборке используя метрику accuracy. Постройте график качества модели на валидационной выборке от количества эпох. Для этого вы можете использовать утилиту tensorboard.
<ul>
    <li><a href="https://www.tensorflow.org/guide/summaries_and_tensorboard" target="_blank">tensorboard в tensorflow</a></li>
    <li><a href="https://keras.io/callbacks/#tensorboard" target="_blank">tensorboard в keras</a></li>
</ul>

In [6]:
log_reg_path = 'datas/project/logs/log_reg'

def log_reg(x_train, y_train, epochs, validation_data, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(10, activation='softmax', input_shape=(784,)),
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy']
    )
    
    logs= [
        tf.keras.callbacks.TensorBoard(
            log_dir = log_reg_path,
            write_graph=True,
            write_images=True,
            histogram_freq=1,
            profile_batch=100000000
        ),
        
    ]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_data=validation_data,
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs
    )

In [14]:
model = log_reg(x_train, y_train, epochs=100, validation_data=(x_test, y_test), batch_size=50)

Train on 60000 samples, validate on 10000 samples
Epoch 1/100
60000/60000 [==============================] - 5s 80us/sample - loss: 2.2204 - accuracy: 0.4011 - val_loss: 2.1327 - val_accuracy: 0.5618
Epoch 2/100
60000/60000 [==============================] - 4s 67us/sample - loss: 2.0563 - accuracy: 0.6106 - val_loss: 1.9832 - val_accuracy: 0.6147
Epoch 3/100
60000/60000 [==============================] - 4s 63us/sample - loss: 1.9190 - accuracy: 0.6321 - val_loss: 1.8576 - val_accuracy: 0.6290
Epoch 4/100
60000/60000 [==============================] - 4s 67us/sample - loss: 1.8033 - accuracy: 0.6434 - val_loss: 1.7515 - val_accuracy: 0.6435
Epoch 5/100
60000/60000 [==============================] - 4s 70us/sample - loss: 1.7050 - accuracy: 0.6558 - val_loss: 1.6610 - val_accuracy: 0.6570
Epoch 6/100
60000/60000 [==============================] - 4s 61us/sample - loss: 1.6209 - accuracy: 0.6668 - val_loss: 1.5832 - val_accuracy: 0.6650
Epoch 7/100
60000/60000 [=========================

Epoch 55/100
60000/60000 [==============================] - 4s 59us/sample - loss: 0.7789 - accuracy: 0.7669 - val_loss: 0.7823 - val_accuracy: 0.7670
Epoch 56/100
60000/60000 [==============================] - 3s 55us/sample - loss: 0.7750 - accuracy: 0.7679 - val_loss: 0.7785 - val_accuracy: 0.7665
Epoch 57/100
60000/60000 [==============================] - 3s 58us/sample - loss: 0.7712 - accuracy: 0.7685 - val_loss: 0.7748 - val_accuracy: 0.7673
Epoch 58/100
60000/60000 [==============================] - 3s 55us/sample - loss: 0.7675 - accuracy: 0.7692 - val_loss: 0.7712 - val_accuracy: 0.7684
Epoch 59/100
60000/60000 [==============================] - 3s 55us/sample - loss: 0.7639 - accuracy: 0.7693 - val_loss: 0.7676 - val_accuracy: 0.7689
Epoch 60/100
60000/60000 [==============================] - 4s 59us/sample - loss: 0.7604 - accuracy: 0.7706 - val_loss: 0.7642 - val_accuracy: 0.7687
Epoch 61/100
60000/60000 [==============================] - 3s 58us/sample - loss: 0.7571 - ac

In [12]:
os.makedirs(log_reg_path, exist_ok=True)

%reload_ext tensorboard
%tensorboard --logdir=$log_reg_path --host localhost --port=6007

Reusing TensorBoard on port 6007 (pid 2324), started 22:36:00 ago. (Use '!kill 2324' to kill it.)

Модель логистической регрессии показала итоговый результат в <b>78.46%</b> точности

## Полносвязная нейронная сеть

Далее, попробуйте реализовать полносвязную нейронную сеть с несколькими скрытыми слоями. Обучите модель и посчитайте качество на тестовой выборке. Как оно изменилось в сравнении с логистической регрессией? Как вы можете объяснить этот результат?

In [13]:
fcnn_path = 'datas/project/logs/fcnn'

def fcnn(x_train, y_train, epochs, validation_data, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(784, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy']
    )
    
    logs= [
        tf.keras.callbacks.TensorBoard(
            log_dir = fcnn_path,
            write_graph=True,
            write_images=True,
            histogram_freq=1,
            profile_batch=100000000,
        ),
        
    ]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_data=validation_data,
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs
    )

In [33]:
model = fcnn(x_train, y_train, epochs=100, validation_data=(x_test, y_test), batch_size=50)

Train on 60000 samples, validate on 10000 samples
Epoch 1/100
60000/60000 [==============================] - 13s 223us/sample - loss: 2.1233 - accuracy: 0.5096 - val_loss: 1.9112 - val_accuracy: 0.5834
Epoch 2/100
60000/60000 [==============================] - 12s 204us/sample - loss: 1.6749 - accuracy: 0.6095 - val_loss: 1.4650 - val_accuracy: 0.6159
Epoch 3/100
60000/60000 [==============================] - 12s 200us/sample - loss: 1.3175 - accuracy: 0.6575 - val_loss: 1.1975 - val_accuracy: 0.6701
Epoch 4/100
60000/60000 [==============================] - 12s 207us/sample - loss: 1.1074 - accuracy: 0.6858 - val_loss: 1.0361 - val_accuracy: 0.6972
Epoch 5/100
60000/60000 [==============================] - 13s 211us/sample - loss: 0.9741 - accuracy: 0.7059 - val_loss: 0.9285 - val_accuracy: 0.7167
Epoch 6/100
60000/60000 [==============================] - 12s 205us/sample - loss: 0.8826 - accuracy: 0.7225 - val_loss: 0.8531 - val_accuracy: 0.7215
Epoch 7/100
60000/60000 [=============

In [14]:
os.makedirs(fcnn_path, exist_ok=True)

%reload_ext tensorboard
%tensorboard --logdir=$fcnn_path --host localhost --port=6007

Reusing TensorBoard on port 6007 (pid 8572), started 21:52:48 ago. (Use '!kill 8572' to kill it.)

Модель полносвязной нейронной сети показала итоговый результат в <b>85.15%</b>
<hr>
При <i>неизменных параметров запуска</i>, <b>полносвязная нейросеть показывает более точный результат</b>, но <i>тратит на обучение модели больше времени</i>.

## Сверточная нейронная сеть

После этого вам предлагается реализовать сверточную нейронную сеть. В данном случае лучше использовать готовые слои, которые предоставляют keras или tensorflow.

Начните с модели с несколькими сверточными слоями. Так же рекомендуется использовать слои суб-дискретизации, например Max Pooling слои. Они понижают размерность сходных данных и выделяют наиболее важные признаки из данных. Посчитайте качество получившейся модели на тестовой выборке. Сравните полученные результаты с результатами полносвязной нейронной сети.

In [15]:
cnn1_path = 'datas/project/logs/cnn1/'

def cnn1(x_train, y_train, epochs, validation_data, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Convolution2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy']
    )
    
    logs= [
        tf.keras.callbacks.TensorBoard(
            log_dir = cnn1_path,
            write_graph=True,
            write_images=True,
            histogram_freq=1,
            profile_batch=100000000,
        ),
        
    ]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_data=validation_data,
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs
    )

In [9]:
model = cnn1(x_train.values.reshape(60000, 28, 28, 1), 
             y_train, 
             epochs=100, 
             validation_data=(x_test.values.reshape(10000, 28, 28, 1), y_test), 
             batch_size=50
            )

Train on 60000 samples, validate on 10000 samples
Epoch 1/100
60000/60000 [==============================] - 90s 2ms/sample - loss: 2.2284 - accuracy: 0.3659 - val_loss: 1.9109 - val_accuracy: 0.4705
Epoch 2/100
60000/60000 [==============================] - 77s 1ms/sample - loss: 1.0807 - accuracy: 0.6506 - val_loss: 0.8145 - val_accuracy: 0.7030
Epoch 3/100
60000/60000 [==============================] - 65s 1ms/sample - loss: 0.7419 - accuracy: 0.7228 - val_loss: 0.7258 - val_accuracy: 0.7349
Epoch 4/100
60000/60000 [==============================] - 64s 1ms/sample - loss: 0.6697 - accuracy: 0.7471 - val_loss: 0.6380 - val_accuracy: 0.7668
Epoch 5/100
60000/60000 [==============================] - 66s 1ms/sample - loss: 0.6249 - accuracy: 0.7659 - val_loss: 0.6061 - val_accuracy: 0.7779
Epoch 6/100
60000/60000 [==============================] - 67s 1ms/sample - loss: 0.5940 - accuracy: 0.7782 - val_loss: 0.5810 - val_accuracy: 0.7871
Epoch 7/100
60000/60000 [=========================

In [16]:
os.makedirs(cnn1_path, exist_ok=True)

%reload_ext tensorboard
%tensorboard --logdir=$cnn1_path --host localhost --port=6007

Reusing TensorBoard on port 6007 (pid 8036), started 15:53:25 ago. (Use '!kill 8036' to kill it.)

На сверточных сетях итоговое качество составило <b>88.74%</b> при одинаковом кол-ве эпох и параметров. Но время создания модели снова выросло. При чем заметно переобучение за последние 3 эпохи: точность на валидационных данных падала.
<hr>

Далее, попробуйте увеличить количество слоев в вашей нейронной сети. Достаточно добавить несколько новых сверточных слоев. Проанализируете, как изменилось качество в этом случае.

В заключение, рекомендуется попробовать добавить Batch Normalization слои. Обычно они располагаются после сверточных слоев или слоев полносвязной нейронной сети. Обычно они улучшают качество модели, этим объясняется их популярность использования в современных архитектурах нейронных сетей. Однако, это требует проверки для конкретной модели и конкретного набора данных.

In [17]:
cnn2_path = 'datas/project/logs/cnn2/'

def cnn2(x_train, y_train, epochs, validation_data, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Convolution2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(128, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy']
    )
    
    logs= [
        tf.keras.callbacks.TensorBoard(
            log_dir = cnn2_path,
            write_graph=True,
            write_images=True,
            histogram_freq=1,
            profile_batch=100000000,
        ),
        
    ]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_data=validation_data,
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs
    )

In [25]:
model = cnn2(x_train.values.reshape(60000, 28, 28, 1),
             y_train,
             epochs=100,
             validation_data=(x_test.values.reshape(10000, 28, 28, 1), y_test),
             batch_size=50
             )

Train on 60000 samples, validate on 10000 samples
Epoch 1/100
60000/60000 [==============================] - 74s 1ms/sample - loss: 0.9371 - accuracy: 0.6771 - val_loss: 0.6328 - val_accuracy: 0.7707
Epoch 2/100
60000/60000 [==============================] - 70s 1ms/sample - loss: 0.5792 - accuracy: 0.7911 - val_loss: 0.5366 - val_accuracy: 0.8092
Epoch 3/100
60000/60000 [==============================] - 75s 1ms/sample - loss: 0.5120 - accuracy: 0.8141 - val_loss: 0.4854 - val_accuracy: 0.8192
Epoch 4/100
60000/60000 [==============================] - 67s 1ms/sample - loss: 0.4731 - accuracy: 0.8282 - val_loss: 0.4924 - val_accuracy: 0.8234
Epoch 5/100
60000/60000 [==============================] - 69s 1ms/sample - loss: 0.4462 - accuracy: 0.8394 - val_loss: 0.4958 - val_accuracy: 0.8251
Epoch 6/100
60000/60000 [==============================] - 68s 1ms/sample - loss: 0.4284 - accuracy: 0.8447 - val_loss: 0.4774 - val_accuracy: 0.8208
Epoch 7/100
60000/60000 [=========================

Epoch 55/100
60000/60000 [==============================] - 81s 1ms/sample - loss: 0.1748 - accuracy: 0.9362 - val_loss: 0.4575 - val_accuracy: 0.8563
Epoch 56/100
60000/60000 [==============================] - 84s 1ms/sample - loss: 0.1727 - accuracy: 0.9378 - val_loss: 0.5579 - val_accuracy: 0.8294
Epoch 57/100
60000/60000 [==============================] - 84s 1ms/sample - loss: 0.1695 - accuracy: 0.9380 - val_loss: 0.3920 - val_accuracy: 0.8700
Epoch 58/100
60000/60000 [==============================] - 90s 2ms/sample - loss: 0.1682 - accuracy: 0.9387 - val_loss: 0.3735 - val_accuracy: 0.8763
Epoch 59/100
60000/60000 [==============================] - 84s 1ms/sample - loss: 0.1641 - accuracy: 0.9403 - val_loss: 0.4915 - val_accuracy: 0.8496
Epoch 60/100
60000/60000 [==============================] - 81s 1ms/sample - loss: 0.1630 - accuracy: 0.9402 - val_loss: 0.3986 - val_accuracy: 0.8763
Epoch 61/100
60000/60000 [==============================] - 81s 1ms/sample - loss: 0.1580 - ac

In [44]:
os.makedirs(cnn2_path, exist_ok=True)

%reload_ext tensorboard
%tensorboard --logdir=$cnn2_path --host localhost --port=6007

Reusing TensorBoard on port 6007 (pid 956), started 0:51:37 ago. (Use '!kill 956' to kill it.)

Точность второй сверточной сети составила <b>84.25%</b>. Это показывает, что усложнение модели не всегда приводит к улучшению качества.
<hr>

## Ответ

В качестве решения приложите архив, содержащий файл решения и все используемые для его работы файлы.Постройте график качества модели на валидационной выборке от количества эпох. Для этого вы можете использовать утилиту tensorboard.

In [43]:
common_path = 'datas/project/logs/'
os.makedirs(common_path, exist_ok=True)

%tensorboard --logdir=$common_path --host localhost --port=1122

Reusing TensorBoard on port 1122 (pid 788), started 0:02:08 ago. (Use '!kill 788' to kill it.)

При анализе всех графиков точности мы пришли к выводу, что наилучшей точностью обладает валидационная модель <b>cnn1</b>